# Evaluasi temporal PhreshPhish

**[AUTHOR RUN] Jalur 2 dengan lingkup terbatas.** Notebook ini melatih LightGBM dari awal pada bagian latih PhreshPhish dan mengevaluasinya pada tanggal yang lebih akhir. Ini bukan transfer model PhiUSIIL, bukan grid fitur PhiUSIIL, dan bukan replikasi skor probe lama. Dataset mentah tidak disertakan dalam handoff.

Rancangan dibekukan: fitur URL deterministik tanpa label; ambang 0,5; parameter LightGBM dan daftar seed dari `D_probe_code/03_phreshphish_protocol.py`. Lima seed merupakan pengulangan model pada **satu** pembagian waktu, bukan lima periode independen. Tidak ada pemilihan model, ambang, atau fitur berdasarkan hasil uji. Karena probe pada bagian uji pernah dilihat, analisis ini tetap eksploratif.

Pembagian yang dimodifikasi membuang baris latih pada atau setelah 2025-09-08, mempertahankan bagian uji sejak tanggal itu, serta mengeluarkan URL uji yang sudah terdapat pada data latih. Semua perubahan jumlah baris dicatat. Domain boleh berulang lintas waktu; notebook mengukur keterpaparan domain aktual pada bagian uji.

Jalankan berurutan. Ubah `DATA_DIR` dan `OUTPUT_ROOT` pada sel konfigurasi. Model tidak mengunduh atau membuka halaman URL. Hasil penelitian baru hanya tersedia setelah notebook dijalankan pada korpus penulis.

In [ ]:
# [AUTHOR RUN] C0 — dependensi
import importlib.util, subprocess, sys
packages = {'numpy':'numpy', 'pandas':'pandas', 'pyarrow':'pyarrow',
            'sklearn':'scikit-learn', 'lightgbm':'lightgbm', 'tldextract':'tldextract'}
missing = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

In [ ]:
# [AUTHOR RUN] C1 — konfigurasi sebelum hasil uji dihitung
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, re, time, ipaddress, importlib.metadata
from urllib.parse import urlsplit, urlunsplit
from functools import lru_cache
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
import tldextract
from sklearn.metrics import (confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score, average_precision_score, matthews_corrcoef)

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/PhreshPhish/data')  # UBAH
OUTPUT_ROOT = Path('/content/drive/MyDrive/PhreshPhish_temporal_audit')  # UBAH
CUTOFF = pd.Timestamp('2025-09-08', tz='UTC')
SEEDS = [13, 42, 2024, 777, 31415]
PRIMARY_SEED = 13
THRESHOLD = 0.5
BOOTSTRAP_REPLICATES = 1000  # pilihan rancangan baru, bukan hasil lama
MODEL_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=63,
                    n_jobs=-1, verbosity=-1)
OUT = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT.mkdir(parents=True, exist_ok=False)
ENVIRONMENT = {p: importlib.metadata.version(p) for p in packages.values()}
DESIGN = dict(protocol='modified_temporal_url_deduplicated',
    training_corpus='PhreshPhish', evaluation_corpus='PhreshPhish',
    label_mapping={'benign':0, 'phish':1}, cutoff=str(CUTOFF),
    seeds=SEEDS, primary_seed=PRIMARY_SEED, threshold=THRESHOLD,
    model=MODEL_PARAMS, include_psl_private_domains=False,
    tuning='none', test_status='previously inspected in exploratory probes',
    bootstrap_replicates=BOOTSTRAP_REPLICATES, environment=ENVIRONMENT,
    python=sys.version, platform=platform.platform(), cpu=platform.processor(),
    logical_cpus=os.cpu_count())
(OUT/'design.json').write_text(json.dumps(DESIGN, indent=2), encoding='utf-8')
print('Output:', OUT)
print('Rencana fit:', len(SEEDS), '; tidak ada estimasi durasi dari korpus lain.')

In [ ]:
# [AUTHOR RUN] C2 — fungsi pemuatan dan pemisahan yang dapat diaudit
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def load_partition(root, part):
    paths = sorted(Path(root).glob(f'{part}-*.parquet'))
    if not paths:
        raise FileNotFoundError(f'Tidak ada {part}-*.parquet di {root}')
    frames, manifest = [], []
    for path in paths:
        schema = pq.read_schema(path)
        needed = {'url','label','date'}
        if not needed.issubset(schema.names):
            raise ValueError(f'Skema berbeda: {path.name}; diperlukan {needed}')
        frame = pq.read_table(path, columns=['url','label','date']).to_pandas()
        frame['source_file'] = path.name
        frame['source_row'] = np.arange(len(frame))
        frames.append(frame)
        manifest.append(dict(file=path.name, rows=len(frame), bytes=path.stat().st_size,
                             sha256=file_sha256(path)))
    df = pd.concat(frames, ignore_index=True)
    if df[['url','label','date']].isna().any().any():
        raise ValueError(f'{part}: URL/label/tanggal kosong. Audit sumber diperlukan.')
    if not df.url.map(lambda u: isinstance(u,str) and bool(u.strip())).all():
        raise ValueError(f'{part}: URL tidak berupa teks valid.')
    if not set(df.label.unique()).issubset({'benign','phish'}):
        raise ValueError(f'{part}: label tidak dikenal: {df.label.unique()}')
    df['y_phish'] = df.label.map({'benign':0,'phish':1}).astype('int8')
    df['date'] = pd.to_datetime(df.date, errors='raise', utc=True)
    if df.date.isna().any():
        raise ValueError(f'{part}: tanggal tidak valid.')
    df['row_id'] = part + ':' + df.source_file + ':' + df.source_row.astype(str)
    return df, manifest

def parse_url(u):
    u = u.strip()
    p = urlsplit(u if '://' in u else 'http://' + u)
    if p.scheme.lower() not in {'http','https'} or not p.hostname:
        raise ValueError('URL memerlukan host dan skema HTTP(S) yang dapat diurai.')
    _ = p.port  # menolak port tidak valid
    host = p.hostname.lower().rstrip('.')
    if any(c.isspace() for c in host):
        raise ValueError('Spasi dalam host.')
    return p, host

def canonical_url(u):
    p, host = parse_url(u)
    # Host dan skema tidak peka huruf besar. Path/query tetap utuh.
    # Fragmen tidak dikirim ke server; kredensial dipertahankan bila ada.
    host_netloc = '[' + host + ']' if ':' in host else host
    if p.port is not None:
        host_netloc += ':' + str(p.port)
    auth = p.netloc.rsplit('@',1)[0] + '@' if '@' in p.netloc else ''
    return urlunsplit((p.scheme.lower(),auth+host_netloc,p.path or '/',p.query,''))

def prepare_temporal(train_raw, test_raw, cutoff):
    ledger = []
    def record(stage, df):
        ledger.append(dict(stage=stage, rows=len(df),
            benign=int((df.y_phish==0).sum()), phish=int((df.y_phish==1).sum()),
            min_date=str(df.date.min()), max_date=str(df.date.max())))
    train, test = train_raw.copy(), test_raw.copy()
    record('train_raw',train); record('test_raw',test)
    if (test.date < cutoff).any():
        raise ValueError('Bagian test memiliki tanggal sebelum batas; jangan pindahkan otomatis.')
    excluded = train.loc[train.date >= cutoff, ['row_id']].copy()
    excluded['reason'] = 'train_at_or_after_cutoff'
    train = train.loc[train.date < cutoff].copy()
    record('train_after_boundary_purge',train)
    for name, df in [('train',train),('test',test)]:
        keys, bad = [], []
        for row in df.itertuples(index=False):
            try:
                keys.append(canonical_url(row.url))
            except (ValueError, UnicodeError):
                keys.append(None); bad.append(row.row_id)
        if bad:
            raise ValueError(f'{name}: {len(bad)} URL tidak dapat diurai; '
                             f'periksa row_id {bad[:5]} dan dokumentasikan keputusan.')
        df['url_key'] = keys
    # Training tidak diubah berdasarkan label test.
    conflict = train.groupby('url_key').y_phish.nunique().gt(1)
    if conflict.any():
        raise ValueError('Label konflik pada URL latih; adjudikasi sebelum fitting.')
    train = train.sort_values(['date','row_id'], kind='stable')
    duplicate = train.duplicated('url_key', keep='first')
    ex = train.loc[duplicate,['row_id']].copy(); ex['reason']='duplicate_train_url'
    excluded = pd.concat([excluded,ex],ignore_index=True)
    train = train.loc[~duplicate].copy()
    record('train_unique',train)
    repeated = test.url_key.isin(set(train.url_key))
    ex = test.loc[repeated,['row_id']].copy(); ex['reason']='test_url_seen_in_train'
    excluded = pd.concat([excluded,ex],ignore_index=True)
    test = test.loc[~repeated].copy()
    record('test_after_seen_url_purge',test)
    conflict = test.groupby('url_key').y_phish.nunique().gt(1)
    if conflict.any():
        raise ValueError('Label konflik pada URL uji; adjudikasi tanpa mengubah training.')
    test = test.sort_values(['date','row_id'], kind='stable')
    duplicate = test.duplicated('url_key',keep='first')
    ex = test.loc[duplicate,['row_id']].copy(); ex['reason']='duplicate_test_url'
    excluded = pd.concat([excluded,ex],ignore_index=True)
    test = test.loc[~duplicate].copy()
    record('test_unique',test)
    for name, df in [('train',train),('test',test)]:
        if set(df.y_phish.unique()) != {0,1}:
            raise ValueError(f'{name}: kedua kelas harus tersedia setelah penyaringan.')
    assert train.date.max() < test.date.min(), 'Urutan waktu tidak ketat.'
    assert set(train.row_id).isdisjoint(test.row_id), 'Baris bertumpang-tindih.'
    assert set(train.url_key).isdisjoint(test.url_key), 'URL bertumpang-tindih.'
    return (train.reset_index(drop=True), test.reset_index(drop=True),
            pd.DataFrame(ledger), excluded)

In [ ]:
# [AUTHOR RUN] C3 — audit data, tanpa fitting
tr_raw, tr_manifest = load_partition(DATA_DIR, 'train')
te_raw, te_manifest = load_partition(DATA_DIR, 'test')
(OUT/'input_manifest.json').write_text(json.dumps(tr_manifest+te_manifest,indent=2),encoding='utf-8')
train, test, ledger, excluded = prepare_temporal(tr_raw,te_raw,CUTOFF)
ledger.to_csv(OUT/'row_ledger.csv',index=False)
excluded.to_csv(OUT/'excluded_rows.csv',index=False)
print(ledger.to_string(index=False))
print('Ukuran setelah pembersihan adalah hasil baru; jangan salin ukuran mentah ke tabel hasil.')
del tr_raw, te_raw

In [ ]:
# [AUTHOR RUN] C4 — fitur URL deterministik, tanpa label dan tanpa pengambilan HTML
EXT = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=False)
SPEC = re.compile(r'[^A-Za-z0-9]')
@lru_cache(maxsize=None)
def host_parts(host):
    try:
        ipaddress.ip_address(host)
        return host, '', '', True
    except ValueError:
        e = EXT(host)
        reg = e.domain + '.' + e.suffix if e.suffix else (e.domain or host)
        return reg, e.subdomain or '', e.suffix or '', False

def url_features(u):
    u = u.strip()
    p,host = parse_url(u)
    reg,sub,suffix,is_ip = host_parts(host)
    length = len(u) or 1
    letters = sum(c.isalpha() for c in u)
    digits = sum(c.isdigit() for c in u)
    obfuscated = len(re.findall(r'%[0-9A-Fa-f]{2}',u))
    special = len(SPEC.findall(u))
    return {'URLLength':length, 'DomainLength':len(reg), 'TLDLength':len(suffix),
        'NoOfSubDomain':len([s for s in sub.split('.') if s]),
        'NoOfObfuscatedChar':obfuscated, 'ObfuscationRatio':obfuscated/length,
        'NoOfLettersInURL':letters, 'LetterRatioInURL':letters/length,
        'NoOfDegitsInURL':digits, 'DegitRatioInURL':digits/length,
        'NoOfEqualsInURL':u.count('='), 'NoOfQMarkInURL':u.count('?'),
        'NoOfAmpersandInURL':u.count('&'), 'NoOfOtherSpecialCharsInURL':special,
        'SpacialCharRatioInURL':special/length, 'IsHTTPS':int(p.scheme.lower()=='https'),
        'HasObfuscation':int(obfuscated>0), 'IsDomainIP':int(is_ip)}

def domain_for(u):
    _,host = parse_url(u)
    return host_parts(host)[0]

t0 = time.perf_counter()
Xtrain = pd.DataFrame([url_features(u) for u in train.url],dtype='float32')
train_extract_s = time.perf_counter()-t0
t0 = time.perf_counter()
Xtest = pd.DataFrame([url_features(u) for u in test.url],dtype='float32')
test_extract_s = time.perf_counter()-t0
assert list(Xtrain.columns)==list(Xtest.columns)
assert 'TLDLegitimateProb' not in Xtrain
assert np.isfinite(Xtrain.to_numpy()).all() and np.isfinite(Xtest.to_numpy()).all()
train['domain'] = train.url.map(domain_for)
test['domain'] = test.url.map(domain_for)
test['domain_seen'] = test.domain.isin(set(train.domain))
for part,df in [('train',train),('test',test)]:
    export = df[['row_id','date','y_phish','domain']].copy()
    export['url_sha256'] = df.url_key.map(lambda u: hashlib.sha256(u.encode()).hexdigest())
    if part=='test': export['domain_seen']=df.domain_seen
    export.to_csv(OUT/f'split_{part}.csv.gz',index=False)
exposure = test.groupby('y_phish').domain_seen.agg(['size','sum','mean']).reset_index()
exposure.to_csv(OUT/'actual_test_domain_exposure.csv',index=False)
(OUT/'feature_schema.json').write_text(json.dumps(dict(names=list(Xtrain.columns),
    count=len(Xtrain.columns), source='deterministic URL-only; no target encoding',
    semantics='New extractor; not asserted identical to PhiUSIIL precomputed columns',
    train_extract_s=train_extract_s, test_extract_s=test_extract_s),indent=2),encoding='utf-8')
print('Fitur:',len(Xtrain.columns)); print(exposure.to_string(index=False))
print('Eksposur di atas memakai domain data latih, bukan frekuensi domain dalam korpus gabungan.')

In [ ]:
# [AUTHOR RUN] C5 — fitting dan pencatatan setiap seed
def metrics(y,prob):
    y=np.asarray(y); prob=np.asarray(prob)
    if not len(y):
        return {'n':0}
    pred=(prob>=THRESHOLD).astype('int8')
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    both=len(np.unique(y))==2
    return dict(n=len(y), n_benign=int(tn+fp), n_phish=int(tp+fn),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
        prevalence=float(y.mean()), f1_phish=f1_score(y,pred,zero_division=0),
        precision_phish=precision_score(y,pred,zero_division=0),
        recall_phish=recall_score(y,pred,zero_division=0),
        fpr_benign=float(fp/(tn+fp)) if tn+fp else np.nan,
        miss_phish=float(fn/(tp+fn)) if tp+fn else np.nan,
        roc_auc=roc_auc_score(y,prob) if both else np.nan,
        average_precision=average_precision_score(y,prob) if both else np.nan,
        mcc=matthews_corrcoef(y,pred) if both else np.nan)

results=[]
for seed in SEEDS:
    model=lgb.LGBMClassifier(**MODEL_PARAMS,random_state=seed)
    t0=time.perf_counter(); model.fit(Xtrain,train.y_phish)
    fit_s=time.perf_counter()-t0
    phish_column=int(np.flatnonzero(model.classes_==1)[0])
    t0=time.perf_counter(); prob=model.predict_proba(Xtest)[:,phish_column]
    prediction_s=time.perf_counter()-t0
    prediction_ms_per_row=prediction_s/len(test)*1000
    model.booster_.save_model(str(OUT/f'model_seed_{seed}.txt'))
    predfile=test[['row_id','date','y_phish','domain','domain_seen']].copy()
    predfile['p_phish']=prob
    predfile.to_csv(OUT/f'predictions_seed_{seed}.csv.gz',index=False)
    for stratum,mask in [('all',np.ones(len(test),dtype=bool)),
                         ('seen_domain',test.domain_seen.to_numpy()),
                         ('unseen_domain',~test.domain_seen.to_numpy())]:
        r=dict(seed=seed,stratum=stratum,**metrics(test.y_phish.to_numpy()[mask],prob[mask]))
        # Waktu untuk seluruh batch, tidak diatribusikan ke subset.
        if stratum=='all':
            r.update(fit_s=fit_s,batch_predict_s=prediction_s,
                     amortized_predict_ms_per_row=prediction_ms_per_row)
        results.append(r)
    pd.DataFrame(results).to_csv(OUT/'results_per_seed.csv',index=False)
    (OUT/f'model_params_seed_{seed}.json').write_text(
        json.dumps(model.get_params(),indent=2,default=str),encoding='utf-8')
    print('Selesai seed',seed,'fit_s=',fit_s)
R=pd.DataFrame(results)
summary=R.groupby('stratum')[['f1_phish','precision_phish','recall_phish',
    'fpr_benign','miss_phish','roc_auc','average_precision','mcc']].agg(['mean','std','min','max'])
summary.to_csv(OUT/'seed_variation.csv')
print(summary.to_string())
print('SD antar-seed bukan ketidakpastian lintas periode. Jangan menyebutnya CI temporal.')

In [ ]:
# [AUTHOR RUN] C6 — ketidakpastian berbasis domain pada model seed utama, tanpa fitting baru
def rates_from_confusion(c):
    tn,fp,fn,tp=map(float,c)
    def div(a,b): return a/b if b else np.nan
    return {'f1_phish':div(2*tp,2*tp+fp+fn), 'precision_phish':div(tp,tp+fp),
            'recall_phish':div(tp,tp+fn), 'fpr_benign':div(fp,fp+tn)}

def cluster_bootstrap(frame,reps,seed):
    y=frame.y_phish.to_numpy(); pred=(frame.p_phish.to_numpy()>=THRESHOLD).astype(int)
    c=pd.DataFrame({'domain':frame.domain.to_numpy(),
        'tn':((y==0)&(pred==0)).astype(int), 'fp':((y==0)&(pred==1)).astype(int),
        'fn':((y==1)&(pred==0)).astype(int), 'tp':((y==1)&(pred==1)).astype(int)})
    counts=c.groupby('domain')[['tn','fp','fn','tp']].sum().to_numpy()
    if len(counts)<2:
        raise ValueError('Domain terlalu sedikit untuk bootstrap cluster.')
    rng=np.random.default_rng(seed); samples=[]
    for _ in range(reps):
        idx=rng.integers(0,len(counts),size=len(counts))
        samples.append(rates_from_confusion(counts[idx].sum(axis=0)))
    B=pd.DataFrame(samples); point=rates_from_confusion(counts.sum(axis=0))
    return pd.DataFrame([dict(metric=k,estimate=v,lower95=B[k].quantile(.025),
        upper95=B[k].quantile(.975),valid_replicates=int(B[k].notna().sum())) for k,v in point.items()])

primary=pd.read_csv(OUT/f'predictions_seed_{PRIMARY_SEED}.csv.gz')
ci=cluster_bootstrap(primary,BOOTSTRAP_REPLICATES,PRIMARY_SEED)
ci.to_csv(OUT/'cluster_bootstrap_primary_seed.csv',index=False)
primary['month']=pd.to_datetime(primary.date,utc=True).dt.strftime('%Y-%m')
monthly=[]
for month,frame in primary.groupby('month'):
    monthly.append(dict(month=month,seed=PRIMARY_SEED,**metrics(frame.y_phish,frame.p_phish)))
pd.DataFrame(monthly).to_csv(OUT/'monthly_descriptive_primary_seed.csv',index=False)
files=[dict(file=p.name,sha256=file_sha256(p),bytes=p.stat().st_size)
       for p in sorted(OUT.iterdir()) if p.is_file() and p.name!='output_manifest.json']
(OUT/'output_manifest.json').write_text(json.dumps(files,indent=2),encoding='utf-8')
print(ci.to_string(index=False))
print('CI adalah resampling domain pada satu periode uji dan satu model tetap; bukan jaminan generalisasi temporal.')
print('Selesai. Kirim row_ledger.csv, results_per_seed.csv, cluster_bootstrap_primary_seed.csv,')
print('actual_test_domain_exposure.csv, design.json, feature_schema.json, dan kedua manifest untuk penelaahan.')

Interpretasi setelah eksekusi: laporkan skor dan ketidakpastian sebagaimana dihasilkan, termasuk hasil yang tidak mendukung dugaan penurunan kinerja. Jangan menghitung “degradasi temporal” dengan mengurangkan skor PhreshPhish ini dari Tabel 2 PhiUSIIL atau dari probe random yang tercemar. Model, fitur, prevalensi, dan korpus berbeda. Jika skor tetap tinggi, hasil tersebut tetap merupakan evaluasi pada observasi lebih akhir, bukan bukti bahwa penuaan korpus tidak ada.

Simpan semua keluaran bersama notebook dan versi pustaka. Fitur `TLDLegitimateProb` dihapus karena implementasi probe memakai rerata label menurut panjang TLD di luar partisi latih. Ekstraktor baru menghasilkan fitur URL dengan definisi eksplisit; kesamaan nama tidak menjamin kesamaan semantik dengan fitur prahitung PhiUSIIL. Indikator IP memakai `ipaddress`; HTTPS memeriksa skema secara tepat. Domain memakai snapshot PSL yang dibundel pada versi tldextract tercatat, tanpa permintaan jaringan dan tanpa memasukkan sufiks privat.

Rujukan implementasi (bukan penambahan daftar pustaka naskah): [scikit-learn tentang kebocoran](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage), [tldextract](https://github.com/john-kurkowski/tldextract), dan [LightGBM](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html).